# 11 Customer deployment
Point your retrieval at a second, simulated customer, run the leak test with their roles, and keep a field log.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-11"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: read the customer brief

In [ ]:
import json, re
import pandas as pd
from kit.llm import call_llm

# Northwind Learning is invented practice content: an online tutoring firm with three roles.
CUSTOMER = Path(os.environ.get("CUSTOMER_DIR", ROOT / "data" / "customer_northwind"))
brief = (CUSTOMER / "BRIEF.md").read_text()
print("\n".join(brief.splitlines()[:24]))

Step 3: re-point retrieval at their pages, with their roles

In [ ]:
from kit.rag import VectorIndex, chunk_pages, get_embedder, parse_corpus

# Their roles, not Quillnote's: who may read a page with each visibility.
NW_ACCESS = {"student": ["student", "tutor", "admin"], "tutor": ["tutor", "admin"], "admin": ["admin"]}
pages = parse_corpus(CUSTOMER / "pages")
chunks = chunk_pages(pages, 120, 30, access=NW_ACCESS)
index = VectorIndex(chunks, get_embedder())
print(len(pages), "pages,", len(chunks), "chunks, embedder:", index.embedder.name)
pd.DataFrame(pages)[["id", "title", "visibility"]]

Step 4: answer their questions as each role

In [ ]:
SYSTEM = ("Answer using only the numbered sources. Cite the source ids you used in square brackets, like [nw-02#0]. "
          "If the sources do not answer the question, say so and tell the user to contact support.")
questions = [json.loads(line) for line in open(CUSTOMER / "questions.jsonl")]

def answer(question, role):
    hits = index.search(question, k=3, role=role)  # the role filter from Chapter 9, with their roles
    sources = "\n".join(f"[{c['id']}] {c['text']}" for c, _ in hits)
    r = call_llm(f"Sources:\n{sources}\n\nQuestion: {question}", system=SYSTEM, max_output_tokens=1000)
    cited = {c.split("#")[0] for c in re.findall(r"nw-\d+(?:#\d+)?", r["text"])}
    return r["text"], cited, [c["page_id"] for c, _ in hits]

rows = []
for q in questions:
    text, cited, retrieved = answer(q["question"], q["role"])
    rows.append({"qid": q["qid"], "role": q["role"], "question": q["question"], "expected": q["page_ids"],
                 "retrieved": retrieved, "cited": sorted(cited), "correct": bool(cited & set(q["page_ids"])),
                 "answer": text[:100]})
results = pd.DataFrame(rows)
print(results.groupby("role", sort=False).correct.agg(["mean", "count"]).round(2))
results[~results.correct][["qid", "role", "question", "expected", "retrieved"]]

Step 5: the leak test, with the new roles

In [ ]:
VISIBILITY = {p["id"]: p["visibility"] for p in pages}
probes = [q["question"] for q in questions]
probes += ["tutor pay rate", "refund a late cancel", "safeguarding report details", "deactivate a tutor"]

def seen(role, k=5):
    return {VISIBILITY[c["page_id"]] for q in probes for c, _ in index.search(q, k=k, role=role)}

for role in ["student", "tutor", "admin", None]:
    print(f"{str(role):8} sees pages marked: {sorted(seen(role))}")
assert seen("student") == {"student"}, "LEAK: a student retrieved a tutor or admin page"
assert "admin" not in seen("tutor"), "LEAK: a tutor retrieved an admin page"
assert "admin" in seen(None), "the test must be able to fail: with no filter, admin pages come back"
print("No leaks with the role filter on.")

Step 6: fill the field log

In [ ]:
from datetime import date

DECISIONS = {"configure", "build", "drop"}  # change settings or content / a product change / not worth it
log = pd.read_csv(CUSTOMER / "field_log.csv")  # the template: the header only
print("Columns:", list(log.columns))

today = date.today().isoformat()
entries = []
for row in results[~results.correct].itertuples():
    entries.append({"date": today, "issue": f"{row.role} question not answered: {row.question}",
                    "workaround": "support answers it by email for now", "decision": "configure",
                    "note": f"expected {row.expected}, retrieved {row.retrieved}: check that page's visibility, "
                            "or write a page for this role"})
# Two things you would hear on a customer call, written up the same way.
entries += [
    {"date": today, "issue": "Parents ask questions but there is no parent role", "workaround": "parents use the student login",
     "decision": "drop", "note": "out of pilot scope; ask again after week 4"},
    {"date": today, "issue": "Admins want each answer to link to its help page", "workaround": "page ids shown as text",
     "decision": "build", "note": "a product change: clickable sources in the widget, for every customer"},
]
assert all(e["decision"] in DECISIONS for e in entries), "decision must be configure, build or drop"
log = pd.concat([log, pd.DataFrame(entries)], ignore_index=True)
log.to_csv(OUTPUT / "field_log.csv", index=False)
print("Wrote", len(log), "rows to", OUTPUT.name + "/field_log.csv")
log